# Soluciones · 00 ¿Qué mide un VNA?

Soluciones resueltas de los ejercicios del [cuaderno 00](../../notebooks/00_introduccion_vna.ipynb). Intenta cada ejercicio por tu cuenta antes de leer su solución: el intento es donde se aprende.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vnacourse import plotting

plotting.use_course_style()

c0 = 299_792_458.0
v = 0.7 * c0
z0 = 50.0


def reflection(z_load):
    return (z_load - z0) / (z_load + z0)

## 1. ¿Cuándo es corto un cable?

Una longitud $\ell$ es una décima de longitud de onda cuando $\lambda = 10\,\ell$, es decir, a $f = v / (10\,\ell)$.

In [ ]:
for name, length in (("cable de 10 cm", 0.10), ("patilla de 1 cm", 0.01)):
    f_tenth = v / (10 * length)
    print(f"{name}: una décima de longitud de onda a {f_tenth / 1e6:.0f} MHz")

print(f"\nA 1 GHz la longitud de onda en el cable es de {v / 1e9 * 100:.0f} cm")
print(f"así que una patilla de 1 cm es 1/{v / 1e9 / 0.01:.0f} de longitud de onda")

Un cable de 10 cm deja de ser «solo un cable» hacia los 210 MHz, y la patilla de 1 cm de un componente hacia los 2.1 GHz. A 1 GHz una patilla de 1 cm ya es una vigesimoprimera parte de la longitud de onda: todavía no es una línea de transmisión, pero ya no es despreciable. A estas frecuencias, las patillas, el conector y la forma de montar la pieza son parte de lo que mides. Por eso los componentes de RF son diminutos y por eso importa el *plano de referencia* del cuaderno 03.

## 2. Circuito abierto

Un abierto tiene $\Gamma = +1$. Añádelo a la gráfica de onda estacionaria.

In [ ]:
f = 300e6
beta = 2 * np.pi * f / v
d = np.linspace(0, 1.0, 500)

fig, ax = plt.subplots()
for name, gamma in (("cortocircuito  (Γ = −1)", -1.0), ("abierto  (Γ = +1)", 1.0)):
    ax.plot(d, np.abs(1 + gamma * np.exp(-2j * beta * d)), label=name)
ax.set(xlabel="distancia a la carga (m)", ylabel="amplitud de la tensión, $|V| / |V^+|$")
ax.set_title("Ondas estacionarias de un cortocircuito y un abierto a 300 MHz")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)

wavelength = v / f
print(f"Longitud de onda: {wavelength:.2f} m, cuarto de onda: {wavelength / 4:.3f} m")

Los dos patrones tienen la misma forma, desplazados un **cuarto de longitud de onda** (0.175 m). En la propia carga, el cortocircuito obliga a que la tensión sea cero, mientras que el abierto tiene un máximo de tensión (y corriente nula). El primer cero del abierto está un cuarto de longitud de onda delante de él.

Una consecuencia que conviene recordar: un circuito abierto visto a través de un cuarto de longitud de onda de línea parece un cortocircuito, y al revés.

## 3. De pérdidas de retorno a impedancia

Pérdidas de retorno $= -20\log_{10}|\Gamma|$, así que $|\Gamma| = 10^{-RL/20}$. Para una carga puramente resistiva, $\Gamma = \pm|\Gamma|$, y $R = Z_0 (1 + \Gamma)/(1 - \Gamma)$.

In [ ]:
return_loss_db = 10.0
gamma_max = 10 ** (-return_loss_db / 20)
r_low = z0 * (1 - gamma_max) / (1 + gamma_max)
r_high = z0 * (1 + gamma_max) / (1 - gamma_max)

print(f"Mayor |Γ| permitido:             {gamma_max:.3f}")
print(f"Potencia reflejada, como mucho:  {gamma_max**2:.0%}")
print(f"Intervalo de impedancia resistiva: de {r_low:.1f} Ω a {r_high:.1f} Ω")

Unas pérdidas de retorno de 10 dB permiten $|\Gamma|$ de hasta 0.316: un 10 % de la potencia reflejada y un 90 % entregada. Si la impedancia es puramente resistiva, está entre 26 Ω y 96 Ω. Los dos límites son $Z_0/1.92$ y $1.92\,Z_0$: la especificación es simétrica como *cociente* respecto a 50 Ω, no como diferencia. «Mejor que 10 dB» es un requisito holgado, y por eso es tan habitual en las hojas de datos de antenas.

## 4. Un cable más largo

Con un cable más largo, la onda reflejada acumula fase más deprisa con la frecuencia.

In [ ]:
f_hz = np.linspace(50e3, 1.45e9, 401)

fig, ax = plt.subplots()
for length in (0.20, 1.0):
    tau = length / v
    s11 = reflection(75.0) * np.exp(-4j * np.pi * f_hz * tau)
    phase = np.unwrap(np.angle(s11))
    ax.plot(f_hz / 1e6, np.degrees(phase), label=f"{length:g} m de cable")

    slope = np.polyfit(f_hz, phase, 1)[0]  # rad por Hz
    tau_estimated = -slope / (4 * np.pi)
    print(
        f"{length:g} m: retardo a partir de la pendiente de la fase {tau_estimated * 1e9:.2f} ns, "
        f"longitud / v = {tau * 1e9:.2f} ns"
    )
ax.set(xlabel="frecuencia (MHz)", ylabel="fase desenrollada de $S_{11}$ (°)")
ax.set_title("Fase de la reflexión para dos longitudes de cable")
ax.legend(loc="lower left");

La fase es una recta cuya pendiente es proporcional a la longitud del cable: cinco veces más largo, cinco veces más inclinada. De $\angle S_{11} = -4\pi f\tau$, el retardo en un sentido es $\tau = -\text{pendiente}/(4\pi)$, y coincide exactamente con $\ell/v$.

Dos notas prácticas. `np.unwrap` elimina los saltos de ±180°, pero solo si la fase cambia menos de 180° entre puntos vecinos: un cable largo necesita una rejilla de frecuencias densa. Y este es exactamente el método que se usa con datos reales en el cuaderno 03 para encontrar la línea oculta delante de una resistencia.